In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, roc_curve
)
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data_path = Path('data/analise_emocional_completa.csv')
df = pd.read_csv(data_path)
df.dropna(inplace=True)

In [ ]:
contagem_cols = ['contagem_raiva','contagem_medo','contagem_tristeza','contagem_alegria','contagem_surpresa']
for col in contagem_cols:
    df[f'{col}_log'] = np.log1p(df[col])
    df[f'{col}_prop'] = df[col] / df['total_palavras']


In [ ]:
feature_cols = [
    'media_sensitivity','media_attitude','media_temper','media_introspection',
    # contagens brutas
    *contagem_cols,
    # contagens log e proporções
    *[f'{c}_log' for c in contagem_cols],
    *[f'{c}_prop' for c in contagem_cols],
]
X = df[feature_cols]
y = df['label'].map({'fake': 1, 'true': 0})

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        random_state=42,
        max_iter=1000,
        class_weight='balanced'
    )),
])

pipe_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(
        random_state=42,
        class_weight='balanced'
    )),
])

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_grid_lr = {
    'clf__C': [0.01, 0.1, 1, 10],
    'clf__penalty': ['l2']
}

param_grid_rf = {
    'clf__n_estimators': [100, 200],
    'clf__max_depth': [None, 10, 20]
}

grid_lr = GridSearchCV(
    pipe_lr, param_grid_lr,
    cv=cv, scoring='f1', n_jobs=-1, verbose=1
)

grid_rf = GridSearchCV(
    pipe_rf, param_grid_rf,
    cv=cv, scoring='f1', n_jobs=-1, verbose=1
)

In [ ]:
print(">> Treinando Logistic Regression")
grid_lr.fit(X_train, y_train)
print("Melhores parâmetros LR:", grid_lr.best_params_)
print("F1 CV (LR):", grid_lr.best_score_)

print("\n>> Treinando Random Forest")
grid_rf.fit(X_train, y_train)
print("Melhores parâmetros RF:", grid_rf.best_params_)
print("F1 CV (RF):", grid_rf.best_score_)

In [ ]:
best_lr = grid_lr.best_estimator_
best_rf = grid_rf.best_estimator_

for name, model in [('Logistic Regression', best_lr), ('Random Forest', best_rf)]:
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:,1]
    print(f"\n=== Avaliação: {name} ===")
    print("Acurácia:", accuracy_score(y_test, y_pred))
    print("F1-score:", f1_score(y_test, y_pred))
    print("ROC AUC:", roc_auc_score(y_test, y_proba))
    print("\nRelatório de Classificação:")
    print(classification_report(y_test, y_pred))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['true','fake'], yticklabels=['true','fake'])
    plt.title(f'Matriz de Confusão - {name}')
    plt.xlabel('Previsto'); plt.ylabel('Verdadeiro')
    plt.show()

    # Curva ROC
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc_score(y_test,y_proba):.2f})')
    plt.plot([0,1],[0,1],'--', color='gray')
    plt.title(f'Curva ROC - {name}')
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
    plt.legend()
    plt.show()

In [ ]:
voting = VotingClassifier([
    ('lr', best_lr),
    ('rf', best_rf)
], voting='soft', n_jobs=-1)

voting.fit(X_train, y_train)
y_pred_v = voting.predict(X_test)
print("\n=== Avaliação: Ensemble Voting ===")
print("F1-score:", f1_score(y_test, y_pred_v))
print("ROC AUC:", roc_auc_score(y_test, voting.predict_proba(X_test)[:,1]))